# 02 · Agents and the Runner

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

Notebook 01 ran agents. This one looks closely at the two objects you'll use
in every program: the **Agent** (what the model is told and given) and the
**Runner** (how the loop runs and when it stops). You'll also meet the
**context** object, the SDK's way of passing your own data (user, database,
settings) into tools and instructions without the model seeing it.

**You will learn**
- Every important `Agent` setting, and **dynamic instructions**
- The Runner's three entry points: `run`, `run_sync`, `run_streamed`
- The exact rules for **when a run ends**, and `max_turns`
- Multi-turn conversations by hand with `to_input_list()`
- **Context**: dependency injection for your own code, invisible to the model
- `RunConfig`: settings for a whole run

**Prerequisites:** notebook 01.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

from dataclasses import dataclass
from agents import Agent, Runner, function_tool, RunContextWrapper, RunConfig, MaxTurnsExceeded
from common import get_model

model = get_model()

## 1 · Why these two objects

Separating **what an agent is** (Agent) from **running it** (Runner) is a
deliberate design:

- The same agent can be run many times, concurrently, for different users,
  because it holds no per-run data.
- Per-run data (the conversation, the user, the database connection) travels
  with the **run**, as the input and the context.
- An agent can be copied and tweaked with `agent.clone(...)` without affecting the original.

## 2 · The Agent: what the model is told and given

| Setting | Meaning |
|---|---|
| `name` | identifies the agent (in traces, handoffs, `result.last_agent`) |
| `instructions` | the system prompt: a string, **or a function** that builds it per run |
| `model`, `model_settings` | which model, and settings like temperature (notebook 03) |
| `tools` | functions or agents it can call (notebook 04) |
| `handoffs` | agents it can hand the conversation to (notebook 06) |
| `input_guardrails`, `output_guardrails` | checks that can stop the run (notebook 07) |
| `output_type` | a type the final answer must have (notebook 03) |
| `tool_use_behavior` | what happens after a tool runs (notebook 04) |
| `handoff_description` | how *other* agents see this one when deciding to hand off |

**Dynamic instructions.** `instructions` can be a function of the run's context
and the agent. It's called at the start of every model call, so the prompt can
depend on who the user is, the date, feature flags, and so on:

In [ ]:
@dataclass
class UserInfo:                      # our own data, passed into the run as "context"
    name: str
    level: str                       # 'beginner' or 'expert'


def tutor_instructions(ctx: RunContextWrapper[UserInfo], agent: Agent) -> str:
    user = ctx.context
    style = 'simple words and an analogy' if user.level == 'beginner' else 'precise technical terms'
    return f'You are a tutor talking to {user.name}. Answer in two sentences using {style}.'


tutor = Agent[UserInfo](name='Tutor', instructions=tutor_instructions, model=model)

for user in (UserInfo('Ada', 'beginner'), UserInfo('Grace', 'expert')):
    result = await Runner.run(tutor, 'What is a reducer in a workflow engine?', context=user)
    print(f'{user.name} ({user.level}):', result.final_output, '\n')

One agent definition, two users, two different system prompts, built per run
from the context.

## 3 · The Runner: three ways to run

| Method | Returns | Use for |
|---|---|---|
| `await Runner.run(agent, input)` | `RunResult` when finished | most code, notebooks, servers |
| `Runner.run_sync(agent, input)` | `RunResult` | plain scripts (**not** inside Jupyter or an async app) |
| `Runner.run_streamed(agent, input)` | `RunResultStreaming` immediately; events arrive as it runs | UIs (notebook 10) |

`input` is either a string (one user message) or a **list of items** (a whole
conversation, see §5). It can also be a saved `RunState` (notebook 08).

## 4 · When does a run end?

This is the most important thing to understand about the Runner. After every
model call it checks, in this order:

```
 model responded
   │
   ├─ the agent has an output_type and the response is a valid object of it ──► FINAL OUTPUT
   ├─ no output_type and the response is plain text with no tool calls ───────► FINAL OUTPUT
   ├─ the response contains a handoff ──► switch to the new agent, loop again
   ├─ the response contains tool calls ──► run them, then:
   │        tool_use_behavior == 'run_llm_again' (default) ──► loop again
   │        tool_use_behavior says stop ──► the tool result is the FINAL OUTPUT   (notebook 04)
   └─ turns used == max_turns ──► raise MaxTurnsExceeded
```

A **turn** is one model call. `max_turns` (default **10**) is the loop guard:
when a run needs more model calls than that, it's cut off. Here's a task that
genuinely needs many *sequential* turns, because each tool call depends on the
previous result, run with `max_turns=4`:

In [ ]:
@function_tool
def next_step(n: int) -> int:
    """Return the step after n."""
    return n + 1


counter = Agent(name='Counter', model=model, tools=[next_step],
                instructions='Start at 0. Call next_step with the current number, then again with the '
                             'result, one call at a time, until the result is 10. Never skip steps.')
try:
    result = await Runner.run(counter, 'Go.', max_turns=4)
    print('finished:', result.final_output)
except MaxTurnsExceeded as err:
    print('stopped:', err)

The model needed about ten turns and was allowed four, so the Runner raised
`MaxTurnsExceeded` instead of letting it continue. In real agents the same guard
stops a model stuck in a loop (re-checking a status that never changes,
re-searching forever).

**A provider difference to know about.** You can *force* tool use with
`ModelSettings(tool_choice='required')`. OpenAI enforces it, but Ollama's
OpenAI-compatible API **ignores** `tool_choice`: we tested it, and the model just
answered in text. Behaviour that the SDK passes through to the model API
depends on the provider, so test it on the one you use.

Hitting `max_turns` raises an exception, which is a crash, not an answer.
Notebook 12 shows how to turn it into a graceful final output with an error
handler.

## 5 · Conversations: carrying history between runs

Each run is independent: the model only sees what you send. To continue a
conversation, pass the previous run's history plus the new message.
`result.to_input_list()` gives you the whole conversation so far (inputs, tool
calls, outputs) as input items:

In [ ]:
chat = Agent(name='Chat', instructions='You are brief and friendly.', model=model)

r1 = await Runner.run(chat, 'Hi! My name is Ada and I build delivery robots.')
print('turn 1:', r1.final_output)

history = r1.to_input_list() + [{'role': 'user', 'content': 'What do I build?'}]
r2 = await Runner.run(chat, history)
print('turn 2:', r2.final_output)

r3 = await Runner.run(chat, 'What do I build?')          # no history
print('fresh :', r3.final_output)

Managing history by hand is fine for scripts. For real applications, a
**session** stores and reloads it automatically (notebook 09).

## 6 · Context: your data, invisible to the model

The **context** is any Python object you pass as `Runner.run(..., context=obj)`.
Instructions, tools, guardrails and hooks can all read it through a
`RunContextWrapper`. **It is never sent to the model.** That makes it the right
place for:

- the current user and their permissions
- database connections, API clients, loggers
- counters and scratch data your code updates during the run

A tool receives it by declaring a first parameter of type `RunContextWrapper`
(the SDK removes that parameter from the schema the model sees):

In [ ]:
@dataclass
class ShopContext:
    customer_id: str
    orders: dict                 # stands in for a database connection
    lookups: int = 0             # our own counter, updated by tools


@function_tool
def my_orders(ctx: RunContextWrapper[ShopContext]) -> str:
    """List the current customer's orders."""
    ctx.context.lookups += 1
    return str(ctx.context.orders.get(ctx.context.customer_id, []))


shop = Agent[ShopContext](name='Shop assistant', model=model, tools=[my_orders],
                          instructions='Help customers with their orders. Use tools.')

db = {'c-1': ['#101 robot kit (shipped)', '#102 spare battery (processing)'], 'c-2': []}
ctx = ShopContext(customer_id='c-1', orders=db)
result = await Runner.run(shop, 'What have I ordered?', context=ctx)
print(result.final_output)
print('\ntool schema the model saw:', my_orders.params_json_schema)
print('lookups counted in our context:', ctx.lookups)

The model saw a tool **with no parameters**. It couldn't ask for someone else's
orders even if it tried, because the customer id comes from *your* context,
not from the model. That's a real security property: **identity and
permissions belong in context, never in model-controlled arguments.**

## 7 · `RunConfig`: settings for a whole run

`RunConfig` overrides settings for one run without changing the agents: a
different model for every agent in the run, model settings, a workflow name
for tracing, and global guardrails. A typical use is running the same agents on
a cheaper or local model in tests:

In [ ]:
config = RunConfig(model=model, workflow_name='tutoring-demo')
result = await Runner.run(tutor, 'Define "latency" in one sentence.',
                          context=UserInfo('Ada', 'beginner'), run_config=config)
print(result.final_output)

## 8 · Trade-offs

- **Stateless agents are easy to share**, but every run must be given its history
  (or a session) and its context.
- **`max_turns` is a blunt guard.** It counts model calls, not cost. Notebook 12
  adds budgets.
- **Context is powerful and unchecked.** It's plain Python, so tools can mutate it
  freely. Keep it small and explicit (a dataclass), not a grab bag.

## 9 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| `run_sync` in Jupyter / async code | `RuntimeError` about a running event loop | `await Runner.run(...)` |
| Expecting the agent to remember earlier runs | it "forgets" | pass `to_input_list()` history, or a session |
| Putting user ids / secrets in tool arguments | the model can change them | put them in context |
| Putting per-user data on the `Agent` object | data leaks between concurrent runs | pass it as context |
| Huge `max_turns` "to be safe" | runaway cost on a looping model | a small limit, plus an error handler |
| Relying on `tool_choice` to force tool use on Ollama | it's silently ignored | test provider behaviour; enforce order in code instead |

## 10 · Check yourself

<details><summary><b>1.</b> Why is <code>instructions</code> allowed to be a function?</summary>

So the system prompt can be built per run from the context (user name, level, permissions, date, flags) while the agent definition stays shared and stateless.
</details>

<details><summary><b>2.</b> List the ways a run can end.</summary>

A valid `output_type` object, or plain text with no tool calls (when there's no `output_type`); a tool result when `tool_use_behavior` says to stop; or an exception such as `MaxTurnsExceeded` or a guardrail tripwire.
</details>

<details><summary><b>3.</b> What does <code>result.to_input_list()</code> give you, and when do you need it?</summary>

The complete conversation of the run (original input plus every generated item) as input items. You need it to continue the conversation in the next run when you're not using a session.
</details>

<details><summary><b>4.</b> Why pass the customer id through context rather than as a tool argument?</summary>

The model controls tool arguments, so it could request another customer's data (by mistake or through prompt injection). Context is set by your code and never seen or controlled by the model.
</details>

<details><summary><b>5.</b> When would you use <code>RunConfig(model=...)</code>?</summary>

To run existing agents on a different model for one run without editing them, e.g. a cheaper or local model in tests or evaluations.
</details>

## Takeaway

An **Agent** is shared, stateless configuration; the **Runner** executes one run
of the loop, which ends at a final output or raises (e.g. `MaxTurnsExceeded`).
History travels with the input (or a session), and your own data travels in the
**context**, which tools and instructions can use and the model never sees.

Next → `03_models_and_structured_output.ipynb`: connecting models, and getting
typed results instead of text.